# YiGraph 图数据的 Iceberg 建模示例

图数据进入数据湖时，推荐先拆成**顶点表**和**边表**。Iceberg 负责可靠存储与版本管理，YiGraph 负责图查询和图计算。

## 1. 启动 Spark 并创建顶点表

先启动 Spark 会话。项目的 Iceberg、REST Catalog 和 MinIO 参数会从 `spark-defaults.conf` 自动加载。业务主键 `vertex_id` 应长期稳定，`vertex_type` 用于区分人员、企业、设备等顶点类型。

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName('YiGraph图数据建模')
         .getOrCreate())
print(f'Spark {spark.version} 已启动')

Spark 3.5.5 已启动


26/09/17 14:46:10 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lake.yigraph")
spark.sql("""
CREATE OR REPLACE TABLE lake.yigraph.vertices_demo (
  vertex_id STRING,
  vertex_type STRING,
  display_name STRING,
  updated_at TIMESTAMP
) USING iceberg
PARTITIONED BY (vertex_type)
""")
spark.sql("""
INSERT INTO lake.yigraph.vertices_demo VALUES
  ('p-1001', 'person', '张三', TIMESTAMP '2026-09-17 09:00:00'),
  ('p-1002', 'person', '李四', TIMESTAMP '2026-09-17 09:05:00'),
  ('c-2001', 'company', '示例科技', TIMESTAMP '2026-09-17 09:10:00')
""")

DataFrame[]

## 2. 创建边表

每条边明确保存起点、终点和边类型。生产表通常还需要业务时间、来源系统和删除标记。

In [3]:
spark.sql("""
CREATE OR REPLACE TABLE lake.yigraph.edges_demo (
  edge_id STRING,
  src_id STRING,
  dst_id STRING,
  edge_type STRING,
  updated_at TIMESTAMP
) USING iceberg
PARTITIONED BY (edge_type)
""")
spark.sql("""
INSERT INTO lake.yigraph.edges_demo VALUES
  ('e-1', 'p-1001', 'p-1002', 'knows', TIMESTAMP '2026-09-17 10:00:00'),
  ('e-2', 'p-1001', 'c-2001', 'works_at', TIMESTAMP '2026-09-17 10:05:00'),
  ('e-3', 'p-1002', 'c-2001', 'works_at', TIMESTAMP '2026-09-17 10:10:00')
""")

DataFrame[]

## 3. 用 SQL 检查一跳关系

下面把边的两端与顶点表关联。这是导入 YiGraph 前很实用的数据质量检查。

In [4]:
spark.sql("""
SELECT
  src.display_name AS `起点`,
  e.edge_type AS `关系`,
  dst.display_name AS `终点`
FROM lake.yigraph.edges_demo e
LEFT JOIN lake.yigraph.vertices_demo src ON e.src_id = src.vertex_id
LEFT JOIN lake.yigraph.vertices_demo dst ON e.dst_id = dst.vertex_id
ORDER BY e.edge_id
""").show(truncate=False)

+----+--------+--------+
|起点|关系    |终点    |
+----+--------+--------+
|张三|knows   |李四    |
|张三|works_at|示例科技|
|李四|works_at|示例科技|
+----+--------+--------+



## 4. 检查悬空边

边引用了不存在的顶点时，会形成悬空边。正式导入前应确保下面的结果为 0。

In [5]:
spark.sql("""
SELECT COUNT(*) AS dangling_edge_count
FROM lake.yigraph.edges_demo e
LEFT JOIN lake.yigraph.vertices_demo src ON e.src_id = src.vertex_id
LEFT JOIN lake.yigraph.vertices_demo dst ON e.dst_id = dst.vertex_id
WHERE src.vertex_id IS NULL OR dst.vertex_id IS NULL
""").show()

+-------------------+
|dangling_edge_count|
+-------------------+
|                  0|
+-------------------+



## 生产建模建议

1. 顶点和边使用稳定、全局唯一的 ID。
2. 保存 `updated_at`、来源系统和软删除标记，方便增量同步。
3. 高基数字段不要直接作为分区字段；先根据查询和导入批次评估。
4. YiGraph 应通过 Spark/Flink/Trino 或 Iceberg SDK 读取表，不要直接解析 manifest。
5. 用 Iceberg 快照记录每次图数据发布版本，便于回溯和重放。